# Deep Learning Classifiers in GeoWombat

This notebook demonstrates how to use GeoWombat's deep learning classifiers
for land cover classification using the same `fit()` / `predict()` / `fit_predict()`
API as the classical ML classifiers.

**Requirements:** `pip install geowombat[dl]`

This installs: `torch`, `pytorch-tabnet`, `torchgeo`, `segmentation-models-pytorch`

---
## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
from sklearn.preprocessing import LabelEncoder

import geowombat as gw
from geowombat.data import (
    l8_224078_20200518,
    l8_224078_20200518_polygons,
)
from geowombat.ml import fit, fit_predict, predict

### Prepare training labels

Load the polygon labels and encode the land cover classes as integers.
The `col` parameter in `fit()` / `fit_predict()` points to this integer column.

In [ ]:
# Load polygon labels
labels = gpd.read_file(l8_224078_20200518_polygons)
print("Raw labels:")
print(labels.head())

# Encode string classes to integers
le = LabelEncoder()
labels['lc'] = le.fit_transform(labels['name'])
labels = labels.drop(columns=['name'])

print(f"\nClasses: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Number of classes: {len(le.classes_)}")
labels.head()

### Preview the image and labels

In [ ]:
with gw.open(l8_224078_20200518, nodata=0) as src:
    print(f"Image shape: {src.shape}")
    print(f"Image dims:  {src.dims}")
    print(f"CRS:         {src.crs}")
    print(f"Resolution:  {src.res}")
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    src.sel(band=[3, 2, 1]).gw.imshow(mask=True, nodata=0, robust=True, ax=ax)
    labels.plot(ax=ax, column='lc', legend=True, 
                edgecolor='white', linewidth=1, alpha=0.5)
    ax.set_title('Landsat 8 RGB with Training Polygons')
    plt.tight_layout()
    plt.show()

---
## 1. TabNet Classifier

TabNet is an attention-based tabular deep learning model. In GeoWombat, each pixel
is treated as a sample with spectral bands as features (pixel-wise classification).

Key parameters:
- `max_epochs`: Number of training epochs
- `batch_size`: Mini-batch size (default 1024)
- `patience`: Early stopping patience (default 10)
- `verbose`: 0=silent, 1=progress
- `device`: 'cpu', 'cuda', or 'auto'

### 1a. TabNet with `fit_predict()` (one-step)

In [ ]:
from geowombat.ml.dl_classifiers import TabNetClassifier

with gw.config.update(ref_res=150):
    with gw.open(l8_224078_20200518, nodata=0) as src:
        # One-step: fit and predict in a single call
        y = fit_predict(
            src,
            TabNetClassifier(max_epochs=50, verbose=0),
            labels,
            col='lc',
        )

print(f"Prediction shape: {y.shape}")
print(f"Prediction dims:  {y.dims}")
print(f"Unique values:    {np.unique(y.values[np.isfinite(y.values)])}")

In [ ]:
# Plot the TabNet classification result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

with gw.config.update(ref_res=150):
    with gw.open(l8_224078_20200518, nodata=0) as src:
        src.sel(band=[3, 2, 1]).gw.imshow(mask=True, nodata=0, robust=True, ax=axes[0])
        axes[0].set_title('Landsat 8 RGB')

y.sel(band='targ').plot(ax=axes[1], cmap='Set2', add_colorbar=True)
axes[1].set_title('TabNet Classification')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

### 1b. TabNet with separate `fit()` then `predict()`

Use separate steps when you want to inspect the trained model,
save it, or predict on different data.

In [ ]:
with gw.config.update(ref_res=150):
    with gw.open(l8_224078_20200518, nodata=0) as src:
        # Step 1: Fit the model
        clf = TabNetClassifier(max_epochs=50, verbose=0)
        X, Xy, clf = fit(src, clf, labels, col='lc')
        
        print(f"Model fitted: {clf.fitted_}")
        print(f"Number of classes: {clf._n_classes}")
        
        # Step 2: Predict (can use same or different data)
        y = predict(src, X, clf)

print(f"\nPrediction shape: {y.shape}")
vals = y.values[np.isfinite(y.values)]
print(f"Unique predictions: {np.unique(vals)}")

---
## 2. L-TAE Classifier (Temporal Attention)

The Lightweight Temporal Attention Encoder (L-TAE) is designed for satellite
image time series. It uses multi-head temporal attention to classify each pixel
based on its spectral trajectory across multiple dates.

**Key requirement:** Data must have a `time` dimension (opened with `stack_dim='time'`).

Key parameters:
- `n_head`: Number of attention heads (default 4)
- `d_k`: Key dimension per head (default 32)
- `d_model`: Internal embedding dimension (default 128)
- `max_epochs`: Training epochs
- `lr`: Learning rate (default 1e-3)
- `device`: 'cpu', 'cuda', or 'auto'

### 2a. Open multi-temporal data

L-TAE requires time-stacked data. Here we stack the same image twice
to simulate a 2-date time series (in practice, use different acquisition dates).

In [ ]:
from geowombat.ml.dl_classifiers import LTAEClassifier

with gw.config.update(ref_res=150):
    with gw.open(
        [l8_224078_20200518, l8_224078_20200518],
        stack_dim='time',
        nodata=0,
    ) as src:
        print(f"Multi-temporal shape: {src.shape}")
        print(f"Dims: {src.dims}")
        print(f"Time steps: {src.sizes['time']}")
        print(f"Bands: {src.sizes['band']}")

### 2b. L-TAE with `fit_predict()`

In [ ]:
with gw.config.update(ref_res=150):
    with gw.open(
        [l8_224078_20200518, l8_224078_20200518],
        stack_dim='time',
        nodata=0,
    ) as src:
        y = fit_predict(
            src,
            LTAEClassifier(
                max_epochs=50,
                verbose=0,
                d_model=32,   # smaller model for demo
                d_k=8,
                n_head=2,
            ),
            labels,
            col='lc',
        )

print(f"Prediction shape: {y.shape}")
print(f"Prediction dims:  {y.dims}")
print(f"Unique values:    {np.unique(y.values[np.isfinite(y.values)])}")

In [ ]:
# Plot the L-TAE classification result
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

with gw.config.update(ref_res=150):
    with gw.open(l8_224078_20200518, nodata=0) as src:
        src.sel(band=[3, 2, 1]).gw.imshow(mask=True, nodata=0, robust=True, ax=axes[0])
        axes[0].set_title('Landsat 8 RGB')

y.sel(band='targ').plot(ax=axes[1], cmap='Set2', add_colorbar=True)
axes[1].set_title('L-TAE Temporal Classification')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

### 2c. L-TAE with separate `fit()` then `predict()`

In [ ]:
with gw.config.update(ref_res=150):
    with gw.open(
        [l8_224078_20200518, l8_224078_20200518],
        stack_dim='time',
        nodata=0,
    ) as src:
        # Step 1: Fit
        clf = LTAEClassifier(
            max_epochs=50,
            verbose=0,
            d_model=32,
            d_k=8,
            n_head=2,
        )
        X, Xy, clf = fit(src, clf, labels, col='lc')
        
        print(f"Model fitted: {clf.fitted_}")
        print(f"Classes: {clf._n_classes}")
        print(f"Bands: {clf._n_bands}, Timesteps: {clf._n_timesteps}")
        
        # Step 2: Predict
        y = predict(src, X, clf)

print(f"\nPrediction shape: {y.shape}")
vals = y.values[np.isfinite(y.values)]
print(f"Unique predictions: {np.unique(vals)}")

### 2d. L-TAE error handling: no time dimension

L-TAE raises a clear error if data doesn't have a `time` dimension.

In [ ]:
# This will raise a ValueError
try:
    with gw.config.update(ref_res=150):
        with gw.open(l8_224078_20200518, nodata=0) as src:
            clf = LTAEClassifier()
            fit(src, clf, labels, col='lc')
except ValueError as e:
    print(f"Expected error: {e}")

---
## 3. Comparison with Classical ML

The DL classifiers use the exact same `fit()` / `predict()` / `fit_predict()` API
as sklearn-based classifiers. Here's a side-by-side comparison.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

with gw.config.update(ref_res=150):
    with gw.open(l8_224078_20200518, nodata=0) as src:
        # Classical ML: Random Forest
        y_rf = fit_predict(
            src,
            RandomForestClassifier(n_estimators=50, random_state=42),
            labels,
            col='lc',
        )
        
        # Deep Learning: TabNet
        y_tabnet = fit_predict(
            src,
            TabNetClassifier(max_epochs=50, verbose=0),
            labels,
            col='lc',
        )

# Plot side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

with gw.config.update(ref_res=150):
    with gw.open(l8_224078_20200518, nodata=0) as src:
        src.sel(band=[3, 2, 1]).gw.imshow(mask=True, nodata=0, robust=True, ax=axes[0])
        axes[0].set_title('Landsat 8 RGB')

y_rf.sel(band='targ').plot(ax=axes[1], cmap='Set2', add_colorbar=True)
axes[1].set_title('Random Forest')
axes[1].set_aspect('equal')

y_tabnet.sel(band='targ').plot(ax=axes[2], cmap='Set2', add_colorbar=True)
axes[2].set_title('TabNet')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

---
## Notes

- **`n_classes` is auto-inferred** from the label column during `fit()`. You never need to specify it.
- **Feature normalization** is handled internally — raw DN values are standardized before training/prediction.
- **Labels use 1-based encoding** internally (0 = nodata). The classifiers handle the conversion to 0-based for PyTorch and back.
- **`ref_res`** controls the output resolution. Use a coarser resolution (e.g., 150-300m) for faster demos.
- **Device**: Pass `device='cuda'` or `device='auto'` to use GPU acceleration.
- For real workflows, increase `max_epochs` (e.g., 50-200) and use full resolution data.
- L-TAE is most useful with real multi-date imagery where temporal patterns differ between classes.
- This demo uses the same image stacked twice for L-TAE — in practice, use imagery from different dates.